# 🩺 Mini-ChatGPT na nazwach chorób
### Sympozjum „Nowoczesne technologie w pediatrii" · Kamil Jędryczek

Ten notebook buduje **mały model językowy** — dokładnie ten sam mechanizm, na którym opiera się ChatGPT, tylko ~100 miliardów razy mniejszy. Zamiast przewidywać następne **słowo** w zdaniu, przewidujemy następną **literę** w nazwie choroby.

**Jak uruchomić:** menu `Środowisko wykonawcze` → `Uruchom wszystko` (albo Ctrl+F9). Działa też na telefonie.

Trening trwa kilka minut. W tym czasie można czytać komentarze 🙂

In [ ]:
# ── KONFIGURACJA ──────────────────────────────────────────────
# Który zbiór nazw trenujemy? Do wyboru: "choroby", "leki", "dinozaury"
ZBIOR = "choroby"

# Skąd pobrać dane (kolumna nazw, jedna nazwa na linię, bez nagłówka)
DANE = {
    "choroby":   "https://raw.githubusercontent.com/TODO-REPO/sympozjum-pediatria/main/data/choroby.csv",
    "leki":      "https://raw.githubusercontent.com/TODO-REPO/sympozjum-pediatria/main/data/leki.csv",
    "dinozaury": "https://raw.githubusercontent.com/junosuarez/dinosaurs/master/dinosaurs.csv",
}

LICZBA_EPOK = 120   # ile razy model zobaczy cały zbiór podczas nauki

In [ ]:
# ── DANE ──────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv(DANE[ZBIOR], header=None, names=["nazwa"])
nazwy = (
    df["nazwa"].dropna().astype(str).str.strip().str.lower()
      .drop_duplicates().tolist()
)
print(f"Wczytano {len(nazwy)} nazw. Przykłady:")
for n in nazwy[:8]:
    print("  •", n)

## Tokeny specjalne

Każdą nazwę opakowujemy w dwa znaki specjalne:
- `%` — **START**: „zacznij generować"
- `!` — **STOP**: „to już koniec nazwy"

Czyli `pneumonia` staje się `%pneumonia!`. Bez `%` model nie wiedziałby, od czego zacząć; bez `!` generowałby w nieskończoność.

In [ ]:
# ── PRZYGOTOWANIE DANYCH ──────────────────────────────────────
import numpy as np

sekwencje = ["%" + n + "!" for n in nazwy]

# Słownik: każdy znak dostaje numer (0 rezerwujemy na "puste miejsce" / padding)
znaki = sorted(set("".join(sekwencje)))
znak2id = {z: i + 1 for i, z in enumerate(znaki)}
id2znak = {i: z for z, i in znak2id.items()}
WIELKOSC_SLOWNIKA = len(znaki) + 1
print(f"Słownik ma {WIELKOSC_SLOWNIKA} tokenów: {''.join(znaki)}")

# X = nazwa bez ostatniego znaku, Y = nazwa bez pierwszego znaku.
# Model uczy się: po każdym prefiksie przewidź NASTĘPNY znak.
#   X: %pneumonia     Y: pneumonia!
maks_dl = max(len(s) for s in sekwencje)
X = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
Y = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
for i, s in enumerate(sekwencje):
    ids = [znak2id[z] for z in s]
    X[i, :len(ids) - 1] = ids[:-1]
    Y[i, :len(ids) - 1] = ids[1:]
print(f"Macierz treningowa: {X.shape} (nazwy × pozycje znaków)")

## Model

Trzy warstwy, które robią całą magię:
1. **Embedding** — każda litera dostaje swój wektor liczb („znaczenie" litery),
2. **GRU** — sieć rekurencyjna z pamięcią: czyta literę po literze i pamięta kontekst,
3. **Dense + softmax** — zamienia pamięć na **rozkład prawdopodobieństwa** następnej litery.

ChatGPT różni się w środku (Transformer zamiast GRU) i skalą — ale wejście i wyjście ma identyczne: *kontekst → prawdopodobieństwa następnego tokenu*.

In [ ]:
# ── TRENING ───────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Embedding(WIELKOSC_SLOWNIKA, 128, mask_zero=True),
    keras.layers.GRU(256, return_sequences=True),
    keras.layers.Dense(WIELKOSC_SLOWNIKA, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")

historia = model.fit(X, Y, epochs=LICZBA_EPOK, verbose=2)

In [ ]:
# ── KRZYWA UCZENIA ────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(historia.history["loss"])
plt.xlabel("Epoka"); plt.ylabel("Błąd (loss)")
plt.title("Model uczy się: błąd maleje z każdą epoką")
plt.grid(alpha=0.3); plt.show()

## Generowanie — serce autoregresji

Pętla jest banalna:
1. pokaż modelowi dotychczasowy kontekst (zaczynamy od `%`),
2. model zwraca **rozkład prawdopodobieństwa** wszystkich możliwych następnych liter,
3. **wylosuj** literę z tego rozkładu (temperatura steruje „odwagą" losowania),
4. dopisz literę do kontekstu i wróć do punktu 1 — aż wypadnie `!`.

Model **nigdy nie mówi „nie wiem"** — zawsze zwraca jakiś rozkład i zawsze coś wylosujemy. Zapamiętajcie to zdanie przy temacie halucynacji.

In [ ]:
# ── GENEROWANIE ───────────────────────────────────────────────
def generuj(prompt="%", temperatura=1.0, maks_znakow=40, pokaz_kroki=False):
    kontekst = prompt
    for _ in range(maks_znakow):
        ids = np.array([[znak2id[z] for z in kontekst]])
        rozklad = model.predict(ids, verbose=0)[0, -1]          # P(następna litera | kontekst)
        rozklad = np.log(rozklad + 1e-9) / temperatura           # temperatura
        rozklad = np.exp(rozklad) / np.exp(rozklad).sum()
        rozklad[0] = 0; rozklad /= rozklad.sum()                 # nie losujemy paddingu
        wybor = np.random.choice(len(rozklad), p=rozklad)
        znak = id2znak[wybor]
        if pokaz_kroki:
            top5 = np.argsort(rozklad)[::-1][:5]
            opis = "  ".join(f"{id2znak[i]}:{rozklad[i]:.0%}" for i in top5)
            print(f"  kontekst: {kontekst:<25} → wybrano '{znak}'   (top5: {opis})")
        if znak == "!":
            break
        kontekst += znak
    return kontekst[1:]  # bez tokenu startu

print("Jedno generowanie krok po kroku:\n")
nazwa = generuj(pokaz_kroki=True)
print(f"\n✨ Nowa nazwa: {nazwa.upper()}")

In [ ]:
# ── 10 NOWYCH NAZW ────────────────────────────────────────────
for _ in range(10):
    print(" •", generuj(temperatura=1.0))

### Eksport śladu generowania (do slajdu w prezentacji)

Poniższa komórka generuje jedną nazwę i zapisuje wszystkie kroki (rozkłady, wybory) jako JSON w formacie slajdu „Krok po kroku". Uruchamiaj, aż wypadnie nazwa, która Ci się podoba — potem skopiuj JSON.

In [ ]:
# ── ŚLAD GENEROWANIA → JSON POD SLAJD ─────────────────────────
import json

def slad_generowania(temperatura=1.0, maks_znakow=30):
    kontekst, kroki = "%", []
    for _ in range(maks_znakow):
        ids = np.array([[znak2id[z] for z in kontekst]])
        rozklad = model.predict(ids, verbose=0)[0, -1]
        rozklad = np.log(rozklad + 1e-9) / temperatura
        rozklad = np.exp(rozklad) / np.exp(rozklad).sum()
        rozklad[0] = 0; rozklad /= rozklad.sum()
        wybor = int(np.random.choice(len(rozklad), p=rozklad))
        top5 = np.argsort(rozklad)[::-1][:5]
        probs = [[id2znak[int(i)], round(float(rozklad[i]), 2)] for i in top5]
        probs.append(["…", round(max(0.0, 1 - sum(p for _, p in probs)), 2)])
        kroki.append({
            "context": kontekst,
            "probs": probs,
            "chosen": id2znak[wybor],
            "top": id2znak[int(top5[0])],
        })
        if id2znak[wybor] == "!":
            break
        kontekst += id2znak[wybor]
    return kontekst[1:], kroki

nazwa, kroki = slad_generowania()
print(f"✨ Wygenerowano: {nazwa.upper()}  ({len(kroki)} kroków)\n")
print(json.dumps(kroki, ensure_ascii=False, indent=2))

## 📱 Aplikacja dla całej sali (Gradio)

Ostatnia komórka odpala prostą aplikację webową z **publicznym linkiem** (`*.gradio.live`, ważny 72 h). Wystarczy zrobić z niego kod QR — każdy telefon na sali może generować nazwy, bez logowania.

In [ ]:
# ── GRADIO ────────────────────────────────────────────────────
!pip -q install gradio

import gradio as gr

def app_generuj(ile, temperatura):
    wyniki = [generuj(temperatura=temperatura) for _ in range(int(ile))]
    return "\n".join(f"• {w}" for w in wyniki)

demo = gr.Interface(
    fn=app_generuj,
    inputs=[
        gr.Slider(1, 10, value=5, step=1, label="Ile nazw wygenerować?"),
        gr.Slider(0.2, 1.8, value=1.0, step=0.1, label="Kreatywność (temperatura)"),
    ],
    outputs=gr.Textbox(label="Nowe nazwy — właśnie wymyślone przez model", lines=10),
    title=f"🩺 Generator: {ZBIOR}",
    description="Mały model językowy (GRU) wytrenowany na żywo podczas prelekcji. "
                "Ten sam mechanizm co w ChatGPT — tylko mniejszy o ~11 rzędów wielkości.",
)
demo.launch(share=True)